# Futures · Fan-out / Fan-in

**Consigna**
1. Lanzar 5 tareas que simulan llamadas remotas.
2. Recolectar resultados **en orden de finalización**.
3. Agregar una tarea que **falla**.
4. Agregar un **timeout global**.
5. Comparar la implementación con **Future explícito** (`concurrent.futures`) y con **`async/await`** (`asyncio`).

**Discusión:** ¿Qué se gana al desacoplar lanzamiento y recolección de resultados? ¿Dónde conviene poner el manejo de errores?

**Criterio de diseño:** elegir Futures cuando necesitemos *representar, componer o entregar resultados pendientes entre capas*.


In [ ]:
import sys, time, threading, asyncio
from dataclasses import dataclass
from concurrent.futures import (ThreadPoolExecutor, Future, as_completed, wait,
                                TimeoutError as FuturesTimeout)
import pandas as pd

print("Python:", sys.version.split()[0])

# ---- "Llamada remota" simulada: solo espera y puede fallar --------------------------
class ErrorRemoto(Exception):
    pass

def llamada_remota(nombre, demora, falla=False):
    time.sleep(demora)                                   # versión bloqueante (threads)
    if falla:
        raise ErrorRemoto(f"{nombre}: el servidor respondió 500")
    return f"respuesta-{nombre}"

async def llamada_remota_async(nombre, demora, falla=False):
    await asyncio.sleep(demora)                          # versión asíncrona (asyncio)
    if falla:
        raise ErrorRemoto(f"{nombre}: el servidor respondió 500")
    return f"respuesta-{nombre}"

# (nombre, demora en segundos, ¿falla?)  — demoras distintas a propósito:
LLAMADAS = [("A", 1.0, False), ("B", 0.3, False), ("C", 0.7, False),
            ("D", 0.1, False), ("E", 0.5, False)]
print("Orden de lanzamiento:      ", [n for n, _, _ in LLAMADAS])
print("Orden esperado de llegada: ", [n for n, d, _ in sorted(LLAMADAS, key=lambda x: x[1])])

# ---- Reloj para ver QUÉ pasa CUÁNDO -------------------------------------------------
T0 = time.perf_counter()
def reset():
    global T0
    T0 = time.perf_counter()
def reloj():
    return time.perf_counter() - T0

def correr_async(coro):
    """Corre una corrutina en un thread aparte con su propio event loop.
    Colab/Jupyter ya tienen un loop activo, por eso no usamos asyncio.run() directo."""
    res = {}
    def _t():
        res["v"] = asyncio.run(coro)
    t = threading.Thread(target=_t); t.start(); t.join()
    return res["v"]

Python: 3.13.18
Orden de lanzamiento:       ['A', 'B', 'C', 'D', 'E']
Orden esperado de llegada:  ['D', 'B', 'E', 'C', 'A']


## Parte 1 · Fan-out con Futures y fan-in en orden de finalización


In [ ]:
with ThreadPoolExecutor(max_workers=5) as ex:
    reset()
    # FAN-OUT: lanzamos las 5 y NO esperamos a nadie
    futuros = {ex.submit(llamada_remota, n, d, f): n for n, d, f in LLAMADAS}
    print(f"[{reloj():.2f}s] 5 tareas lanzadas. Nadie esperó todavía.")

    un_future = next(iter(futuros))
    print("Un Future recién creado se ve así:", un_future)
    print("¿terminó ya?", un_future.done(), "\n")

    # FAN-IN en orden de FINALIZACIÓN: as_completed entrega cada Future apenas termina
    for f in as_completed(futuros):
        print(f"[{reloj():.2f}s] llegó {futuros[f]} -> {f.result()}")

[0.01s] 5 tareas lanzadas. Nadie esperó todavía.
Un Future recién creado se ve así: <Future at 0x792345590500 state=running>
¿terminó ya? False 

[0.11s] llegó D -> respuesta-D
[0.31s] llegó B -> respuesta-B
[0.51s] llegó E -> respuesta-E
[0.71s] llegó C -> respuesta-C
[1.00s] llegó A -> respuesta-A


Para ver por qué importa el **orden de finalización**, comparamos con recolectar en el **orden de lanzamiento** (esperar primero a `A`, que es la más lenta).

In [ ]:
filas = []
with ThreadPoolExecutor(max_workers=5) as ex:
    reset()
    futs = [(n, ex.submit(llamada_remota, n, d, f)) for n, d, f in LLAMADAS]
    print("Recolectando en ORDEN DE LANZAMIENTO:")
    for n, f in futs:
        f.result()
        print(f"  [{reloj():.2f}s] entregado {n}")
        filas.append(("Orden de lanzamiento", n, round(reloj(), 2)))

with ThreadPoolExecutor(max_workers=5) as ex:
    reset()
    futs = {ex.submit(llamada_remota, n, d, f): n for n, d, f in LLAMADAS}
    print("\nRecolectando en ORDEN DE FINALIZACIÓN:")
    for f in as_completed(futs):
        f.result()
        print(f"  [{reloj():.2f}s] entregado {futs[f]}")
        filas.append(("Orden de finalización", futs[f], round(reloj(), 2)))

df = pd.DataFrame(filas, columns=["Estrategia", "Tarea", "Entregada a los (s)"])
display(df.pivot(index="Tarea", columns="Estrategia", values="Entregada a los (s)"))

Recolectando en ORDEN DE LANZAMIENTO:
  [1.00s] entregado A
  [1.00s] entregado B
  [1.00s] entregado C
  [1.00s] entregado D
  [1.00s] entregado E

Recolectando en ORDEN DE FINALIZACIÓN:
  [0.11s] entregado D
  [0.31s] entregado B
  [0.50s] entregado E
  [0.70s] entregado C
  [1.00s] entregado A


Estrategia,Orden de finalización,Orden de lanzamiento
Tarea,,
A,1.00,1.0
B,0.31,1.0
C,0.70,1.0
D,0.11,1.0
E,0.50,1.0


También se puede pedir al Future que **avise** cuando termine con `add_done_callback`, en vez de esperar nosotros:

In [ ]:
def al_terminar(f):
    print(f"[{reloj():.2f}s] callback corrió en el thread '{threading.current_thread().name}'")

with ThreadPoolExecutor(max_workers=3) as ex:
    reset()
    futs = [ex.submit(llamada_remota, n, d, f) for n, d, f in LLAMADAS[:3]]
    for f in futs:
        f.add_done_callback(al_terminar)      # "avisame cuando termines"
    print(f"[{reloj():.2f}s] main sigue libre mientras tanto...")
    wait(futs)
    time.sleep(0.05)                          # dejamos que terminen de imprimir los callbacks

[0.00s] main sigue libre mientras tanto...
[0.30s] callback corrió en el thread 'ThreadPoolExecutor-3_1'
[0.70s] callback corrió en el thread 'ThreadPoolExecutor-3_2'
[1.00s] callback corrió en el thread 'ThreadPoolExecutor-3_0'


### Conclusión de la parte 1
Lanzar 5 tareas al mismo tiempo hace que todo termine en lo que tarda la **más lenta** (\~1 s) en vez de la suma (~2,6 s). Pero el momento en que *recolectamos* también importa: si esperamos en el orden en que lanzamos, la tarea lenta `A` bloquea la entrega de las demás aunque ya estuvieran listas. Si recolectamos **a medida que van terminando**, la primera respuesta llega a los 0,1 s y podemos empezar a trabajar con ella enseguida. El Future es lo que hace posible ese desacople: es un "ticket" que podemos consultar cuando y como queramos.

## Parte 2 · Agregar una tarea que falla
Sumamos la tarea `F`, que a los 0,4 s lanza una excepción. Un Future **guarda** la excepción: no explota en el thread que la produjo (que moriría en silencio), sino que **espera en el Future** hasta que alguien la pida con `.result()` (la relanza) o `.exception()` (la devuelve).

Primero, la forma ingenua:

In [ ]:
LLAMADAS_F = LLAMADAS + [("F", 0.4, True)]

with ThreadPoolExecutor(max_workers=6) as ex:
    reset()
    futs = {ex.submit(llamada_remota, n, d, f): n for n, d, f in LLAMADAS_F}
    try:
        for f in as_completed(futs):
            print(f"[{reloj():.2f}s] llegó {futs[f]} -> {f.result()}")
    except ErrorRemoto as e:
        print(f"[{reloj():.2f}s] el primer error cortó TODO el fan-in: {e}")
        print("    Quedaron sin recolectar: ", [futs[f] for f in futs if not f.done()],
              "(siguen ejecutándose, pero nadie mira sus resultados)")

[0.10s] llegó D -> respuesta-D
[0.30s] llegó B -> respuesta-B
[0.41s] el primer error cortó TODO el fan-in: F: el servidor respondió 500
    Quedaron sin recolectar:  ['A', 'C', 'E'] (siguen ejecutándose, pero nadie mira sus resultados)


Ese diseño pierde resultados buenos por culpa de uno malo. La alternativa es **convertir cada Future en un resultado explícito** (`ok` / `error`) en el punto de recolección:

In [ ]:
@dataclass
class Resultado:
    nombre: str
    estado: str      # "ok" | "error" | "timeout"
    detalle: str
    t: float         # segundo en que lo supimos

def fan_in(futs, t0):
    """Recolecta TODO en orden de finalización; los errores son datos, no excepciones."""
    out = []
    for f in as_completed(futs):
        exc = f.exception()              # devuelve la excepción (o None); NO la relanza
        out.append(Resultado(futs[f],
                             "error" if exc else "ok",
                             str(exc) if exc else f.result(),
                             round(time.perf_counter() - t0, 2)))
    return out

with ThreadPoolExecutor(max_workers=6) as ex:
    t0 = time.perf_counter()
    futs = {ex.submit(llamada_remota, n, d, f): n for n, d, f in LLAMADAS_F}
    resultados = fan_in(futs, t0)

display(pd.DataFrame([r.__dict__ for r in resultados]))
print("Éxitos:", sum(r.estado == "ok" for r in resultados), "| Errores:", sum(r.estado == "error" for r in resultados))

,nombre,estado,detalle,t
0,D,ok,respuesta-D,0.1
1,B,ok,respuesta-B,0.3
2,F,error,F: el servidor respondió 500,0.4
3,E,ok,respuesta-E,0.5
4,C,ok,respuesta-C,0.7
5,A,ok,respuesta-A,1.0


Éxitos: 5 | Errores: 1


### Conclusión de la parte 2
Cuando una de las tareas falla, la pregunta clave es **qué política queremos**: ¿cortar todo ante el primer error (*fail-fast*) o juntar lo que se pueda y reportar los fallos (*resultados parciales*)? Un Future permite ambas porque **guarda la excepción como un valor** hasta que alguien decida qué hacer con ella. En el primer intento, el error cortó el fan-in y perdimos los resultados buenos; en el segundo, cada resultado quedó etiquetado como `ok` o `error` y nada se perdió.

## Parte 3 · Timeout global


In [ ]:
ex = ThreadPoolExecutor(max_workers=6)
TIMEOUT = 0.8
t0 = time.perf_counter()
futs = {ex.submit(llamada_remota, n, d, f): n for n, d, f in LLAMADAS_F}

llegada = {}                                               # cuándo terminó cada Future
for f in futs:
    f.add_done_callback(lambda f: llegada.__setitem__(f, round(time.perf_counter() - t0, 2)))

hechos, pendientes = wait(futs, timeout=TIMEOUT)          # <- el timeout GLOBAL
t_corte = time.perf_counter() - t0

resultados = []
for f in hechos:
    exc = f.exception()
    resultados.append(Resultado(futs[f], "error" if exc else "ok", str(exc) if exc else f.result(), llegada.get(f, round(t_corte, 2))))
for f in pendientes:
    se_pudo = f.cancel()                                   # solo cancela si NO empezó a correr
    resultados.append(Resultado(futs[f], "timeout", f"cancel() devolvió {se_pudo}", round(t_corte, 2)))

ex.shutdown(wait=False, cancel_futures=True)               # no esperar a los que siguen corriendo
display(pd.DataFrame([r.__dict__ for r in resultados]).sort_values("nombre").reset_index(drop=True))
print(f"Respuesta armada a los {t_corte:.2f} s (timeout = {TIMEOUT} s)")

print("Threads vivos justo después del corte:", threading.active_count())
time.sleep(0.5)
print("Threads vivos 0,5 s después:          ", threading.active_count(),
      " ← la tarea 'timeout' siguió corriendo en segundo plano hasta terminar sola")

,nombre,estado,detalle,t
0,A,timeout,cancel() devolvió False,0.8
1,B,ok,respuesta-B,0.3
2,C,ok,respuesta-C,0.7
3,D,ok,respuesta-D,0.1
4,E,ok,respuesta-E,0.5
5,F,error,F: el servidor respondió 500,0.4


Respuesta armada a los 0.80 s (timeout = 0.8 s)
Threads vivos justo después del corte: 6
Threads vivos 0,5 s después:           5  ← la tarea 'timeout' siguió corriendo en segundo plano hasta terminar sola


Hay que tener en cuenta que al usar `with ThreadPoolExecutor(...)` el timeout no funciona. Al salir del `with`, Python hace `shutdown(wait=True)` y **espera a todas las tareas que seguían corriendo**:

In [ ]:
t0 = time.perf_counter()
with ThreadPoolExecutor(max_workers=6) as ex:
    futs = {ex.submit(llamada_remota, n, d, f): n for n, d, f in LLAMADAS_F}
    hechos, pendientes = wait(futs, timeout=0.8)
    print(f"Dentro del with: timeout cumplido a los {time.perf_counter()-t0:.2f} s, pendientes: {[futs[f] for f in pendientes]}")
print(f"Salí del with a los {time.perf_counter()-t0:.2f} s  ← el with esperó a las pendientes; el timeout 'no funcionó'")

Dentro del with: timeout cumplido a los 0.80 s, pendientes: ['A']
Salí del with a los 1.00 s  ← el with esperó a las pendientes; el timeout 'no funcionó'


### Conclusión de la parte 3
Un timeout global nos deja entregar **algo a tiempo** en lugar de quedarnos esperando a la tarea más lenta: aprovechamos lo que llegó (`B`, `C`, `D`, `E`), marcamos lo que falló (`F`) y declaramos `A` como `timeout`.

**No existe forma de matar un thread**. `cancel()` solo sirve para tareas que todavía no arrancaron; las que ya corrían siguen hasta terminar, y un `with` las espera. El timeout controla *cuánto esperamos nosotros*, no *cuánto trabaja el sistema*.

## Parte 4 · La misma estrategia con `async/await`
En `asyncio`, `asyncio.create_task(...)` cumple el rol de `submit`: lanza la corrutina y devuelve un **Task** (que es un tipo de Future). Armamos las dos versiones como funciones equivalentes para compararlas con los mismos datos.

In [ ]:
# ---------- Versión A: Futures explícitos (threads) ----------
def fan_threads(llamadas, timeout):
    t0 = time.perf_counter()
    ex = ThreadPoolExecutor(max_workers=len(llamadas))
    futs = {ex.submit(llamada_remota, n, d, f): n for n, d, f in llamadas}          # fan-out
    pend, out = set(futs), []
    try:
        for f in as_completed(futs, timeout=timeout):                                # fan-in
            pend.discard(f)
            exc = f.exception()
            out.append(Resultado(futs[f], "error" if exc else "ok",
                                 str(exc) if exc else f.result(), round(time.perf_counter() - t0, 2)))
    except FuturesTimeout:
        pass
    for f in pend:                                                                   # timeout global
        out.append(Resultado(futs[f], "timeout",
                             "cancelada" if f.cancel() else "sigue corriendo en su thread",
                             round(time.perf_counter() - t0, 2)))
    ex.shutdown(wait=False, cancel_futures=True)
    return out, time.perf_counter() - t0

# ---------- Versión B: async/await ----------
async def fan_async(llamadas, timeout):
    t0 = time.perf_counter()
    tareas = {asyncio.create_task(llamada_remota_async(n, d, f)): n for n, d, f in llamadas}   # fan-out
    pend, out = set(tareas), []
    fin = t0 + timeout
    while pend:                                                                      # fan-in
        restante = fin - time.perf_counter()
        if restante <= 0:
            break
        hechas, pend = await asyncio.wait(pend, timeout=restante,
                                          return_when=asyncio.FIRST_COMPLETED)
        for t in hechas:
            exc = t.exception()
            out.append(Resultado(tareas[t], "error" if exc else "ok",
                                 str(exc) if exc else t.result(), round(time.perf_counter() - t0, 2)))
    for t in pend:                                                                   # timeout global
        t.cancel()                                  # cancelación REAL: la tarea recibe CancelledError
        out.append(Resultado(tareas[t], "timeout", "cancelada (CancelledError en su await)",
                             round(time.perf_counter() - t0, 2)))
    await asyncio.gather(*pend, return_exceptions=True)   # dejamos que terminen de cancelarse
    return out, time.perf_counter() - t0

res_a, tot_a = fan_threads(LLAMADAS_F, timeout=0.8)
res_b, tot_b = correr_async(fan_async(LLAMADAS_F, timeout=0.8))

print(f"Futures + threads : respuesta armada en {tot_a:.2f} s")
display(pd.DataFrame([r.__dict__ for r in res_a]))
print(f"async/await       : respuesta armada en {tot_b:.2f} s")
display(pd.DataFrame([r.__dict__ for r in res_b]))

Futures + threads : respuesta armada en 0.80 s


,nombre,estado,detalle,t
0,D,ok,respuesta-D,0.1
1,B,ok,respuesta-B,0.3
2,F,error,F: el servidor respondió 500,0.4
3,E,ok,respuesta-E,0.5
4,C,ok,respuesta-C,0.7
5,A,timeout,sigue corriendo en su thread,0.8


async/await       : respuesta armada en 0.80 s


,nombre,estado,detalle,t
0,D,ok,respuesta-D,0.1
1,B,ok,respuesta-B,0.3
2,F,error,F: el servidor respondió 500,0.4
3,E,ok,respuesta-E,0.5
4,C,ok,respuesta-C,0.7
5,A,timeout,cancelada (CancelledError en su await),0.8


Y para ver **por qué el fan-in manual vale la pena**, la forma más corta en `asyncio` — `gather` — con timeout:

In [ ]:
async def con_gather():
    corrutinas = [llamada_remota_async(n, d, f) for n, d, f in LLAMADAS_F]

    # (a) gather + return_exceptions: los errores pasan a ser valores... pero el orden es el de LANZAMIENTO
    t0 = time.perf_counter()
    r = await asyncio.gather(*corrutinas, return_exceptions=True)
    print(f"(a) gather completo, a los {time.perf_counter()-t0:.2f} s, en orden de lanzamiento:")
    for (n, _, _), x in zip(LLAMADAS_F, r):
        print("    ", n, "->", type(x).__name__ if isinstance(x, Exception) else x)

    # (b) gather con timeout global: TODO o NADA
    corrutinas = [llamada_remota_async(n, d, f) for n, d, f in LLAMADAS_F]
    try:
        await asyncio.wait_for(asyncio.gather(*corrutinas, return_exceptions=True), timeout=0.8)
    except asyncio.TimeoutError:
        print("(b) wait_for(gather, 0.8) -> TimeoutError: se perdieron TAMBIÉN los resultados que sí habían llegado")
correr_async(con_gather())

(a) gather completo, a los 1.00 s, en orden de lanzamiento:
     A -> respuesta-A
     B -> respuesta-B
     C -> respuesta-C
     D -> respuesta-D
     E -> respuesta-E
     F -> ErrorRemoto
(b) wait_for(gather, 0.8) -> TimeoutError: se perdieron TAMBIÉN los resultados que sí habían llegado


### Conclusión de la parte 4
Las dos versiones producen **el mismo resultado**: cuatro `ok`, un `error` y un `timeout`, a ~0,8 s. La diferencia está en cómo se cancela lo pendiente. En `asyncio`, `task.cancel()` es **cancelación real**: la tarea recibe `CancelledError` en su próximo `await` y termina enseguida. Con threads, `future.cancel()` falla si la tarea ya está corriendo. Además vimos que los atajos de `asyncio` como `gather` son cómodos, pero entregan en *orden de lanzamiento*, y combinados con `wait_for` son *todo o nada*: para resultados parciales hay que armar el fan-in a mano, igual que con Futures.

## Parte 5 · Comparación: Future explícito vs `async/await`
Primero una prueba de **escala**: 300 llamadas de 0,2 s a la vez.

In [ ]:
N = 300
def escala_threads():
    with ThreadPoolExecutor(max_workers=N) as ex:
        list(ex.map(lambda i: time.sleep(0.2), range(N)))
async def _esc_async():
    await asyncio.gather(*[asyncio.sleep(0.2) for _ in range(N)])
def escala_async():
    correr_async(_esc_async())

def medir(fn):
    t0 = time.perf_counter(); fn(); return time.perf_counter() - t0

t_th, t_as = medir(escala_threads), medir(escala_async)
display(pd.DataFrame({"Enfoque": [f"Futures + {N} threads", f"asyncio ({N} tasks, 1 thread)"],
                      "Tiempo (s)": [round(t_th, 2), round(t_as, 2)]}))

,Enfoque,Tiempo (s)
0,Futures + 300 threads,0.27
1,"asyncio (300 tasks, 1 thread)",0.21



### Conclusión de la parte 5
Para esperas de red **las dos aproximaciones son igual de rápidas**, y a pequeña escala no se nota diferencia. Con cientos o miles de llamadas, `asyncio` es más liviano porque no necesita un thread por tarea.

La elección real depende del **ecosistema**, de la **escala** y de cuánto necesitás **cancelar**.

## Parte 6 · Criterio de diseño: Futures para entregar resultados pendientes entre capas
> *Elegir Futures cuando necesitemos representar, componer o entregar resultados pendientes entre capas.*

Si tuvieramos una app con tres capas: **datos** (llama a servicios remotos), **servicio** (combina datos) y **presentación** (decide cuándo y si espera). Lo que viaja entre capas es un **Future**.

In [ ]:
_pool = ThreadPoolExecutor(max_workers=4)

# --- Capa de DATOS: devuelve un Future, NO bloquea ---------------------------------
def cliente_precio(producto, falla=False) -> Future:
    return _pool.submit(llamada_remota, f"precio-{producto}", 0.3, falla)

def cliente_stock(producto, falla=False) -> Future:
    return _pool.submit(llamada_remota, f"stock-{producto}", 0.5, falla)

# --- Capa de SERVICIO: COMPONE Futures y devuelve OTRO Future ----------------------
def combinar(fa: Future, fb: Future, fn) -> Future:
    resultado = Future()                  # Future "explícito": una promesa que completamos nosotros
    faltan, lock = [2], threading.Lock()

    def _cuando_termina(_):
        with lock:
            faltan[0] -= 1
            if faltan[0] > 0:
                return
        for f in (fa, fb):                # si alguno falló, el error se propaga al Future combinado
            if f.exception() is not None:
                resultado.set_exception(f.exception())
                return
        resultado.set_result(fn(fa.result(), fb.result()))

    fa.add_done_callback(_cuando_termina)
    fb.add_done_callback(_cuando_termina)
    return resultado

def ficha_producto(producto, falla_stock=False) -> Future:
    return combinar(cliente_precio(producto), cliente_stock(producto, falla_stock),
                    lambda p, s: f"{p} + {s}")

# --- Capa de PRESENTACIÓN: decide CUÁNDO esperar y QUÉ hacer con un error -----------
reset()
ficha = ficha_producto("zapatillas")
print(f"[{reloj():.2f}s] servicio devolvió un Future sin bloquear: {ficha}")
print(f"[{reloj():.2f}s] la UI puede hacer otras cosas mientras tanto...")
valor = ficha.result()                    # acá sí se bloquea hasta que esté listo
print(f"[{reloj():.2f}s] ahora sí esperó -> {valor}")
print(f"          (listo a los {reloj():.2f} s ≈ max(0,3; 0,5), no 0,3 + 0,5 = 0,8)\n")

# Caso de error: viaja por las capas dentro del Future y recién lo maneja la presentación
ficha_mala = ficha_producto("zapatillas", falla_stock=True)
try:
    ficha_mala.result()
except ErrorRemoto as e:
    print("La presentación decide qué mostrar ante el error:", e)

[0.00s] servicio devolvió un Future sin bloquear: <Future at 0x7923575479d0 state=pending>
[0.00s] la UI puede hacer otras cosas mientras tanto...
[0.50s] ahora sí esperó -> respuesta-precio-zapatillas + respuesta-stock-zapatillas
          (listo a los 0.50 s ≈ max(0,3; 0,5), no 0,3 + 0,5 = 0,8)

La presentación decide qué mostrar ante el error: stock-zapatillas: el servidor respondió 500


En esta demo se ven las tres palabras del criterio:
- **Representar:** `ficha_producto(...)` devuelve algo que *representa* un resultado que aún no existe.
- **Componer:** `combinar` junta dos Futures en uno nuevo sin bloquear a nadie, y propaga el error si alguno falla.
- **Entregar entre capas:** la capa de datos produce el Future; la de presentación lo consume cuando quiere. Ninguna capa intermedia tuvo que esperar.

Con `async/await` también se puede (devolviendo `Task`s), pero toda la cadena de funciones tiene que ser `async`. Cuando el flujo es **lineal y vive en una sola capa** ("hacé esto, esperá, hacé lo otro"), `async/await` se lee mucho mejor.


# Discusión

## ¿Qué se gana al desacoplar el lanzamiento de la recolección de resultados?

1. **Concurrencia real del tiempo de espera.** Todo se lanza antes de esperar nada; el total es ~el de la más lenta, no la suma (≈ 1 s en vez de ≈ 2,6 s en nuestro ejemplo).
2. **Latencia al primer resultado.** Al recolectar por orden de finalización, el primer resultado útil llegó a los 0,1 s en vez de a los 1,0 s.
3. **Políticas independientes para cada fase.** Cuándo esperar, cuánto esperar (timeout), qué hacer ante un error (cortar o seguir) y si cancelar son decisiones del *fan-in*, no de cada tarea.
4. **Libertad para hacer otra cosa en el medio.** El que lanza no queda bloqueado; puede seguir trabajando o devolver el control.
5. **Composición.** Como el resultado pendiente es un objeto, se puede pasar, combinar (`combinar`), encadenar con callbacks o entregar a otra capa sin esperarlo.
6. **Mejor testeo y control de carga.** Se puede limitar el tamaño del pool, simular Futures ya resueltos y probar el fan-in sin tocar la red.

**Lo que cuesta:** más piezas para razonar (¿quién espera? ¿quién cancela?), errores que aparecen *después* de lanzar, y tareas "huérfanas" que siguen corriendo si nadie las recolecta.



---
# Conclusión final

**Un Future es un placeholder para un resultado que todavía no llegó.** Gracias a ese placeholder podemos separar *lanzar* el trabajo de *esperarlo*, y esa separación nos da tres cosas muy útiles: hacer todo en paralelo (el tiempo total es el de la tarea más lenta, no la suma), atender cada resultado apenas llega (en vez de esperar al más lento primero) y decidir con calma qué hacer con los problemas.

Sobre los **problemas**: una tarea que falla no tiene por qué arruinar el resto. Como el Future guarda la excepción como un dato, podemos juntar los resultados buenos, marcar los malos y decidir al final. Con el **timeout global** pasa algo parecido: devolvemos lo que haya llegado a tiempo en vez de quedarnos esperando a la más lenta. Con la salvedad de que, con threads, "cancelar" no mata lo que ya está corriendo: solo dejamos de esperar.

Sobre **Future explícito vs `async/await`**: para esperas de red rinden parecido. `async/await` es más liviano con miles de tareas y puede cancelar de verdad, pero obliga a que todo el código sea asíncrono. Los Futures con threads funcionan con las librerías de siempre, pero cuestan un thread por tarea y se cancelan peor. Elegimos Futures cuando el resultado pendiente tiene que **viajar, combinarse o entregarse entre capas**; y `async/await` cuando el flujo es secuencial y vive en un solo lugar y el código ya es asíncrono.

Y la regla de diseño para llevarse es: **lanzar todo primero, recolectar a medida que llega, y dejar que el manejo de errores viva donde se conoce la política del conjunto (el fan-in).**